<a href="https://colab.research.google.com/github/kennyaakin/Stable_Diffusion_VAE/blob/main/Stable_Diffusion_VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle
!pip install denoising_diffusion_pytorch uformer-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [2]:
import torch
from torch import nn
from torch.nn import functional as F
import torchvision.models as models
import math
import os
import shutil
from sklearn.model_selection import train_test_split
import os
from google.colab import drive
from denoising_diffusion_pytorch import Unet, GaussianDiffusion
from uformer_pytorch import Uformer
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
! mkdir ~/.kaggle

In [4]:
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json

In [5]:
! chmod 600 ~/.kaggle/kaggle.json

In [6]:
#! kaggle datasets download smaildurcan/turkish-license-plate-dataset
! kaggle datasets download tustunkok/synthetic-turkish-license-plates

Dataset URL: https://www.kaggle.com/datasets/tustunkok/synthetic-turkish-license-plates
License(s): GPL-2.0
 95% 1.48G/1.56G [00:03<00:00, 197MB/s]
100% 1.56G/1.56G [00:03<00:00, 438MB/s]


In [7]:
#! unzip turkish-license-plate-dataset.zip
! unzip synthetic-turkish-license-plates.zip

Streaming output truncated to the last 5000 lines.
  inflating: license-plates/77-A-2266.png  
  inflating: license-plates/77-A-2310.png  
  inflating: license-plates/77-A-2571.png  
  inflating: license-plates/77-A-2671.png  
  inflating: license-plates/77-A-3459.png  
  inflating: license-plates/77-A-3685.png  
  inflating: license-plates/77-A-3705.png  
  inflating: license-plates/77-A-4153.png  
  inflating: license-plates/77-A-4907.png  
  inflating: license-plates/77-A-5753.png  
  inflating: license-plates/77-A-5980.png  
  inflating: license-plates/77-A-6602.png  
  inflating: license-plates/77-A-6951.png  
  inflating: license-plates/77-A-7004.png  
  inflating: license-plates/77-A-7583.png  
  inflating: license-plates/77-A-7850.png  
  inflating: license-plates/77-A-8102.png  
  inflating: license-plates/77-A-8121.png  
  inflating: license-plates/77-A-8188.png  
  inflating: license-plates/77-A-8221.png  
  inflating: license-plates/77-A-8343.png  
  inflating: license-plat

In [8]:
'''# Comment this whole cell out when using the synthetic-turkish-license-plates dataset
# Set your dataset directory
image_dir = "/content/images"  # Replace this with the path where your images are

# Create a new directory for only jpg images (if it doesn't exist)
filtered_image_dir = "/content/filtered_images"
os.makedirs(filtered_image_dir, exist_ok=True)

# List all files in the directory
files = os.listdir(image_dir)

# Filter for only jpg files
for file in files:
    if file.lower().endswith('.jpg'):
        # Copy the .jpg files to the filtered directory
        shutil.copy(os.path.join(image_dir, file), os.path.join(filtered_image_dir, file))

print(f"Filtered .jpg files have been copied to: {filtered_image_dir}")
'''

'# Comment this whole cell out when using the synthetic-turkish-license-plates dataset\n# Set your dataset directory\nimage_dir = "/content/images"  # Replace this with the path where your images are\n\n# Create a new directory for only jpg images (if it doesn\'t exist)\nfiltered_image_dir = "/content/filtered_images"\nos.makedirs(filtered_image_dir, exist_ok=True)\n\n# List all files in the directory\nfiles = os.listdir(image_dir)\n\n# Filter for only jpg files\nfor file in files:\n    if file.lower().endswith(\'.jpg\'):\n        # Copy the .jpg files to the filtered directory\n        shutil.copy(os.path.join(image_dir, file), os.path.join(filtered_image_dir, file))\n\nprint(f"Filtered .jpg files have been copied to: {filtered_image_dir}")\n'

In [9]:
class SelfAttention(nn.Module):
  def __init__(self, n_heads, embd_dim, in_proj_bias=True, out_proj_bias=True):
    super().__init__()
    self.n_heads = n_heads
    self.in_proj = nn.Linear(embd_dim, 3 * embd_dim, bias=in_proj_bias)
    self.out_proj = nn.Linear(embd_dim, embd_dim, bias=out_proj_bias)

    self.d_heads = embd_dim // n_heads

  def forward(self, x, casual_mask=False):
    # x: (batch_size, seq_len, dim)

    batch_size, seq_len, d_emed = x.shape

    interim_shape = (batch_size, seq_len, self.n_heads, self.d_heads)

    # (batch_size, seq_len, dim) -> 3 * (batch_size, seq_len, d_embed)
    q, k, v = self.in_proj(x).chunk(3, dim=-1)

    # change the shape of q, k and v to match the interim shape
    q = q.view(interim_shape)
    k = k.view(interim_shape)
    v = v.view(interim_shape)

    # swap the elements within matrix using transpose
    # take n_heads before seq_len, like that: (batch_size, n_heads, seq_len, d_embed)
    q = q.transpose(1, 2)
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)

    # calculate the attention
    weight = q @ k.transpose(-1, -2)

    if casual_mask:
        # mask where the upper traingle (above the prinicpal dagonal) is 1
        mask = torch.ones_like(weight, dtype=torch.bool).triu(1)
        # fill the upper traingle with -inf
        weight.masked_fill_(mask, -torch.inf)

    weight /= math.sqrt(self.d_heads)

    weight = F.softmax(weight, dim=-1)

    # (batch_size, h_heads, seq_len, dim / h)
    output = weight @ v

    # (batch_size, h_heads, seq_len, dim / h) -> (batch_size, seq_len, n_heads, dim / h)
    output = output.transpose(1, 2)

    # change the shape to the shape of out_proj
    output = output.reshape((batch_size, seq_len, d_emed))

    output = self.out_proj(output)

    return output

In [10]:
class AttentionBlock(nn.Module):
  def __init__(self, channels):
      super().__init__()
      self.groupnorm = nn.GroupNorm(32, channels)
      self.attention = SelfAttention(1, channels)

  def forward(self, x):
      # x: (batch_size, channels, h, w)
      residual = x.clone()

      # (batch_size, channels, h, w) -> (batch_size, channels, h, w)
      x = self.groupnorm(x)

      n, c, h, w = x.shape

      # (batch_size, channels, h, w) -> (batch_size, channels, h * w)
      x = x.view((n, c, h * w))

      # (batch_size, channels, h * w) -> (batch_size, h * w, channels)
      x = x.transpose(-1, -2)

      # perform self-attention without mask
      # (batch_size, h * w, channels) -> (batch_size, h * w, channels)
      x = self.attention(x)

      # (batch_size, h * w, channels) -> (batch_size, channels, h * w)
      x = x.transpose(-1, -2)

      # (batch_size, channels, h * w) -> (batch_size, channels, h, w)
      x = x.view((n, c, h, w))

      # (batch_size, channels, h, w) -> (batch_size, channels, h, w)
      x += residual

      return x

In [11]:
class ResidualBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    self.groupnorm1 = nn.GroupNorm(32, in_channels)
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

    self.groupnorm2 = nn.GroupNorm(32, out_channels)
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

    if in_channels == out_channels:
      self.residual_layer = nn.Identity()
    else:
      self.residual_layer = nn.Conv2d(in_channels, out_channels, kernel_size=1, padding=0)

  def forward(self, x):
    # x: (batch_size, in_channels, h, w)
    residue = x.clone()

    x = self.groupnorm1(x)
    x = F.selu(x)
    x = self.conv1(x)
    x = self.groupnorm2(x)
    x = self.conv2(x)

    return x + self.residual_layer(residue)

In [12]:
#This encoder created residual blocks with too high of a resolution for the T4 GPU to handle, so I scaled it back.
'''class Encoder(nn.Sequential):
    def  __init__(self):
        super().__init__(
            # (batch_size, channel, h, w) -> (batch_size, 128, h, w)
            nn.Conv2d(3, 128, kernel_size=3, padding=1),

            # (batch_size, 128, h, w) -> (batch_size, 128, h, w)
            ResidualBlock(128, 128),

            # (batch_size, 128, h, w) -> (batch_size, 128, h / 2, w / 2)
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=0),

            # (batch_size, 128, h / 2, w / 2) -> (batch_size, 256, h / 2, w / 2)
            ResidualBlock(128, 256),

            # (batch_size, 256, h / 2, w / 2) -> (batch_size, 256, h / 2, w / 2)
            ResidualBlock(256, 256),

            # (batch_size, 256, h / 2, w / 2) -> (batch_size, 256, h / 4, w / 4)
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=0),

            # (batch_size, 256, h / 4, w / 4) -> (batch_size, 512, h / 4, w / 4)
            ResidualBlock(256, 512),

            # (batch_size, 512, h / 4, w / 4) -> (batch_size, 512, h / 4, w / 4)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 4, w / 4) -> (batch_size, 512, h / 8, w / 8)
            nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=0),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            AttentionBlock(512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            nn.GroupNorm(32, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            nn.SiLU(),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 8, h / 8, w / 8)
            nn.Conv2d(512, 8, kernel_size=3, padding=1),

            # (batch_size, 8, h / 8, w / 8) -> (batch_size, 8, h / 8, w / 8)
            nn.Conv2d(8, 8, kernel_size=1, padding=0)
        )
    def forward(self, x):
        # x: (batch_size, channel, h, w)

        for module in self:
            if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
                x = F.pad(x, (0, 1, 0, 1))  # (left, right, top, bottom)
            x = module(x)

        # (batch_size, 8, h / 8, w / 8) -> two tensors of shape (batch_size, 4, h / 8, w / 8)
        mean, log_variance = torch.chunk(x, 2, dim=1)

        # Clamp log variance between -30 and 20
        log_variance = torch.clamp(log_variance, -30, 20)

        # Reparameterization trick
        std = torch.exp(0.5 * log_variance)
        eps = torch.randn_like(std)
        x = mean + eps * std

        # Scale the latent representation
        x *= 0.18215

        return x
'''

class Encoder(nn.Sequential):
    def __init__(self):
        super().__init__(
            # Input: (B, 3, H, W) → (B, 32, H, W)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            ResidualBlock(32, 32),

            # Downsample: (B, 32, H, W) → (B, 64, H/2, W/2)
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            ResidualBlock(64, 64),

            # Downsample: (B, 64, H/2, W/2) → (B, 64, H/4, W/4)
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            ResidualBlock(64, 64),

            # Output: (B, 64, H/4, W/4) → (B, 8, H/4, W/4)
            nn.Conv2d(64, 8, kernel_size=3, padding=1),
            nn.Conv2d(8, 8, kernel_size=1)
        )

    def forward(self, x):
        for module in self:
            x = module(x)

        mean, log_variance = torch.chunk(x, 2, dim=1)
        log_variance = torch.clamp(log_variance, -10, 10)

        return mean, log_variance


In [13]:
#This decoder created residual blocks with too high of a resolution for the T4 GPU to handle, so I scaled it back.
'''class Decoder(nn.Sequential):
    def __init__(self):
        super().__init__(
            # (batch_size, 4, 32, 32) -> (batch_size, 512, 32, 32)
            nn.Conv2d(4, 512, kernel_size=3, padding=1),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            ResidualBlock(512, 512),

            # (batch_Size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            AttentionBlock(512),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            ResidualBlock(512, 512),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            ResidualBlock(512, 512),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            ResidualBlock(512, 512),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 64, 64)
            nn.Upsample(scale_factor=2),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 64, 64)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 64, 64)
            ResidualBlock(512, 512),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 64, 64)
            ResidualBlock(512, 512),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 64, 64)
            ResidualBlock(512, 512),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 128, 128)
            nn.Upsample(scale_factor=2),

            # (batch_size, 512, 128, 128) -> (batch_size, 512, 128, 128)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),

            # (batch_size, 512, 128, 128) -> (batch_size, 256, 128, 128)
            ResidualBlock(512, 256),

            # (batch_size, 256, 128, 128) -> (batch_size, 256, 128, 128)
            ResidualBlock(256, 256),

            # (batch_size, 256, 128, 128) -> (batch_size, 256, 128, 128)
            ResidualBlock(256, 256),

            # (batch_size, 256, 128, 128) -> (batch_size, 256, 256, 256)
            nn.Upsample(scale_factor=2),

            # (batch_size, 256, 256, 256) -> (batch_size, 256, 256, 256)
            nn.Conv2d(256, 256, kernel_size=3, padding=1),

            # (batch_size, 256, 256, 256) -> (batch_size, 128, 256, 256)
            ResidualBlock(256, 128),

            # (batch_size, 128, 256, 256) -> (batch_size, 128, 256, 256)
            ResidualBlock(128, 128),

            # (batch_size, 128, 256, 256) -> (batch_size, 128, 256, 256)
            ResidualBlock(128, 128),

            nn.GroupNorm(32, 128),

            nn.SiLU(),

            # (batch_size, 128, 256, 256) -> (batch_size, 3, 256, 256)
            nn.Conv2d(128, 3, kernel_size=3, padding=1),
        )
    def forward(self, x):
        # x: (batch_size, 4, h / 8, w / 8)

        # remove the scaling adding by the encoder
        x /= 0.18215

        for module in self:
            x = module(x)

        # (batch_size, 3, h, w)
        return x
'''

class Decoder(nn.Sequential):
    def __init__(self):
        super().__init__(
            # Input: (B, 4, H/4, W/4) → (B, 64, H/4, W/4)
            nn.Conv2d(4, 64, kernel_size=3, padding=1),
            ResidualBlock(64, 64),

            # Upsample: (B, 64, H/4, W/4) → (B, 64, H/2, W/2)
            nn.Upsample(scale_factor=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            ResidualBlock(64, 32),

            # Upsample: (B, 32, H/2, W/2) → (B, 32, H, W)
            nn.Upsample(scale_factor=2),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            ResidualBlock(32, 32),

            nn.GroupNorm(8, 32),
            nn.SiLU(),

            # Final: (B, 32, H, W) → (B, 3, H, W)
            nn.Conv2d(32, 3, kernel_size=3, padding=1),
        )

    def forward(self, x):
        x /= 0.18215
        for module in self:
            x = module(x)
        return x



In [14]:
def split_dataset(source_dir, train_dir, test_dir, test_size=0.8, random_state=42):
    image_files = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    train_files, test_files = train_test_split(image_files, test_size=test_size, random_state=random_state)

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    for file in train_files:
        shutil.copy(os.path.join(source_dir, file), os.path.join(train_dir, file))

    for file in test_files:
        shutil.copy(os.path.join(source_dir, file), os.path.join(test_dir, file))

    print(f"Dataset split complete. {len(train_files)} training images, {len(test_files)} test images.")

#source_dir = "./filtered_images" #Comment this when using the turkish-license-plate-dataset
source_dir = "./license-plates" #Uncomment this when using the synthetic-turkish-license-plates dataset
train_dir = "./data/train/images"
test_dir = "./data/test/images"

split_dataset(source_dir, train_dir, test_dir)

Dataset split complete. 20000 training images, 80000 test images.


In [15]:
# Model
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
      mean, logvar = self.encoder(x)
      std = torch.exp(0.5 * logvar)
      eps = torch.randn_like(std)
      z = mean + eps * std
      z *= 0.18215
      decoded = self.decoder(z)
      return decoded, z

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
# from model import Encoder, Decoder

# Device configuration
device = torch.device('cuda')

# Hyperparameters
num_epochs = 8
learning_rate = 1e-4
beta = 0.00025  # KL divergence weight

# Data loading
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
batch_size = 16
dataset = torchvision.datasets.ImageFolder(root='./data/train', transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)


#model = VAE().to(device)
unet = Unet(dim=128, dim_mults=(1,2,4,8))
model = GaussianDiffusion(unet, image_size=128, timesteps=1000).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Add these hyperparameters
accumulation_steps = 1  # Adjust as needed
effective_batch_size = batch_size * accumulation_steps

train_losses = []

# training loop
'''for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for i, (images, _) in enumerate(dataloader):
        images = images.to(device)

        # Forward pass
        reconstructed, encoded = model(images)

        # Compute loss
        recon_loss = nn.L1Loss()(reconstructed, images)

        # Extract mean and log_variance from encoded
        mean, log_variance = torch.chunk(encoded, 2, dim=1)

        kl_div = -0.5 * torch.sum(1 + log_variance - mean.pow(2) - log_variance.exp())
        loss = recon_loss + beta * kl_div

        # Normalize the loss to account for accumulation
        loss = loss / accumulation_steps

        # Backward pass
        loss.backward()

        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * accumulation_steps

        print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(dataloader)}], '
              f'Loss: {loss.item()*accumulation_steps:.4f}, Recon Loss: {recon_loss.item():.4f}, KL Div: {kl_div.item():.4f}')



        with torch.no_grad():
            # Take the first image from the batch
            sample_image = images[0].unsqueeze(0)
            sample_reconstructed = model(sample_image)[0]

            sample_image = (sample_image * 0.5) + 0.5
            sample_reconstructed = (sample_reconstructed * 0.5) + 0.5

            torchvision.utils.save_image(sample_reconstructed, 'reconstructed.png')

    train_losses.append(train_loss / len(dataloader))
  # Save the model checkpoint
    torch.save(model.state_dict(), f'vae_model_epoch_{epoch+1}.pth')

print('Training finished!')'''

for epoch in range(num_epochs):
    model.train()
    for i, (images, _) in enumerate(dataloader):
        images = images.to(device)
        loss = model(images)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch} | Step {i+1}/{len(dataloader)} | Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), f"diffusion_epoch_{epoch+1}.pt")
print("Training complete!")

# 2. Sampling (Insert Here)
model.eval()  # switch to eval mode for sampling
with torch.no_grad():  # no gradients needed
    sampled_imgs = model.sample(batch_size=4)
    torchvision.utils.save_image(
        sampled_imgs,
        'generated.png',
        normalize=True,
        value_range=(-1, 1)  # adjust if your model expects different range
    )

print("Sampled images saved to generated.png")

OutOfMemoryError: CUDA out of memory. Tried to allocate 36.00 MiB. GPU 0 has a total capacity of 22.16 GiB of which 1.38 MiB is free. Process 35365 has 22.15 GiB memory in use. Of the allocated memory 21.89 GiB is allocated by PyTorch, and 41.60 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
!pip install scikit-image

from skimage.metrics import peak_signal_noise_ratio as psnr_metric, structural_similarity as ssim_metric
import numpy as np
from PIL import Image

def calc_metrics(gt_path, gen_path):
    gt = np.array(Image.open(gt_path).convert('RGB'))
    gen = np.array(Image.open(gen_path).convert('RGB'))
    psnr = psnr_metric(gt, gen, data_range=255)
    ssim = ssim_metric(gt, gen, multichannel=True, data_range=255)
    return psnr, ssim

# Example usage
img = Image.open('filtered_images/1.jpg')
# Resize to 128×128
img_resized = img.resize((128, 128), resample=Image.BICUBIC)
# Save the resized image
img_resized.save("high_res_128.png")
gen = 'generated.png'
psnr, ssim = calc_metrics("high_res_128.png", "generated.png")
print(f"📏 PSNR: {psnr:.2f} dB, SSIM: {ssim:.4f}")


ValueError: Input images must have the same dimensions.

In [ ]:
!pip install paddleocr opencv-python-headless

from paddleocr import PaddleOCR
import cv2

ocr = PaddleOCR(use_angle_cls=True, lang='en')
def extract_text(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_OTSU)
    result = ocr.ocr(thresh, cls=True)
    return result[0][1][0] if result else ""

text = extract_text('generated.png')
print("Detected Text:", text)


In [ ]:
import re

province = {f"{i:02d}" for i in range(1, 82)}
forbidden = set()  # Add any disallowed combos here

def check_compliance(text):
    pattern = r'^(0[1-9]|[1-7][0-9]|8[0-1]) [A-Z]{1,3} \d{2,4}$'
    if re.match(pattern, text):
        return text.split()[0] in province and text not in forbidden
    return False

valid = check_compliance(text)
print("Compliance:", valid)


In [ ]:
# After sampling and saving 'generated.png'
psnr, ssim = calc_metrics('gt.png', 'generated.png')
text = extract_text('generated.png')
valid = check_compliance(text)

print(f"PSNR: {psnr:.2f}, SSIM: {ssim:.4f}, OCR Text: '{text}', Compliant: {valid}")


In [ ]:
import matplotlib.pyplot as plt

# plot the loss curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Loss over Time')
plt.legend()
plt.show()

In [ ]:
'''class CustomVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def encode(self, x):
      return self.encoder(x)

    def decode(self, z):
      return self.decoder(z)

    def forward(self, x):
      z = self.encode(x)
      x_reconstructed = self.decode(z)
      return x_reconstructed

    def load_pretrained_weights(self, weight_path):
      self.load_state_dict(torch.load(weight_path))

'''

import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomVAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def encode(self, x):
        mean, logvar = self.encoder(x)
        return mean, logvar

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mean, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mean + eps * std
        z *= 0.18215  # scaling for latent space
        return self.decode(z), z


In [ ]:
'''from diffusers import AutoencoderKL

class DiffuserCompatibleVAE(AutoencoderKL):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae

    def encode(self, x):
      mean, log_var = self.vae.encoder(x)
      print("mean")
      return mean, log_var

    def decode(self, z, **kwargs):
      print("Input shape:", z.shape)
      out = self.vae.decoder(z).unsqueeze(0)
      print(out.shape)
      return out
'''

from diffusers.models.modeling_utils import ModelMixin
from torch.distributions import Normal

class DiffuserCompatibleVAE(ModelMixin):
    def __init__(self, custom_vae):
        super().__init__()
        self.vae = custom_vae

        # Required for SD pipeline initialization
        self.config = type("Config", (), {
            "scaling_factor": 0.18215,
            "block_out_channels": [32, 64, 64]  # Adjust based on your encoder’s structure
        })()

    def encode(self, x, **kwargs):
        mean, logvar = self.vae.encode(x)
        std = torch.exp(0.5 * logvar)
        return {
            "latent_dist": Normal(mean, std)
        }

    def decode(self, z, return_dict=True, **kwargs):
        x = self.vae.decode(z)
        if return_dict:
            return {"sample": x}
        return (x,)



In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("CompVis/stable-diffusion-v1-4", local_dir="./model")

In [ ]:
from diffusers import AutoencoderKL
AutoencoderKL.from_pretrained("./model/vae")

In [ ]:
import torch

weight = torch.load("./vae_model_epoch_1.pth", map_location="cpu")

new_weights = {}
for k, v in weight.items():
  new_key = k.replace("_", "")

  new_key = new_key.replace("residuallayer", "residual_layer").replace("inproj", "in_proj").replace("outproj", "out_proj")
  new_weights[new_key] = v

encoder = Encoder()
decoder = Decoder()

vae = CustomVAE(encoder, decoder)
vae.load_state_dict(new_weights)

device = torch.device("cuda")
vae = vae.to(device)

In [ ]:
vae

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

compatible_vae = DiffuserCompatibleVAE(vae)
pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    vae=compatible_vae,
    safety_checker=None,
    requires_safety_checker=False
).to("cuda")

In [ ]:
prompt = (
    "a photo of a modern car parked on the street with a visible license plate, "
    "taken from the rear, realistic lighting, natural outdoor setting, "
    "photorealistic, ultra high resolution, sharp details, cinematic"
)

'''prompt = (
    "a photo of Turkish license plate, 01 A 2808"
    "photorealistic, ultra high resolution, sharp details, cinematic"
)'''

image = pipe(prompt, num_inference_steps=50).images[0]

In [ ]:
image